# Dense retrieval: BGE-M3 и Giga-480M

Одинаковый эксперимент для двух биэнкодеров: `search_query` → `item_title_raw`, точный FAISS top-K, Recall@10/50/100. Корпус и целевые пары берутся **только из `train.parquet`**. Сплит и выборка тестовых контекстов повторяют `BM25_pipeline.ipynb` и `TFIDF_pipeline.ipynb` (`seed=42`, 1000 контекстов). Это позволяет напрямую сравнить Recall@10.

Ноутбук рассчитан на Kaggle с включённой GPU. В настройках ниже укажите URL репозитория, если он ещё не добавлен в окружение, и путь к датасету, если автообнаружение неоднозначно. Для удалённого клонирования и первой загрузки весов модели Kaggle Internet должен быть включён. Один полный корпус кодируется один раз на модель; FAISS-индекс сохраняется в `/kaggle/working/dense_indexes` для повторного запуска.

## Настройки Kaggle

In [ ]:
REPO_URL = ""  # например, https://github.com/USER/REPO.git; пусто, если код уже доступен
REPO_PATH = None  # например, "/kaggle/working/REPO"; None — найти или клонировать
DATA_PATH = None  # путь к train.parquet; None — автообнаружение в repo/data или /kaggle/input
INSTALL_DEPENDENCIES = True  # False, если зависимости уже установлены
INDEX_DIR = "/kaggle/working/dense_indexes"
FORCE_REBUILD = False  # True — заново закодировать корпус и перезаписать индекс
SEED = 42
N_TEST_QUERIES = 1000  # None — полный test (долго)
KS = (10, 50, 100)
BATCH_SIZE = 16
MAX_LENGTH = 128  # достаточно для эксперимента по заголовкам; одинаково для обеих моделей

## Зависимости и путь к коду

In [ ]:
import subprocess
import sys
from pathlib import Path

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "faiss-cpu>=1.10,<2", "sentence-transformers>=5.1,<6",
        "transformers>=4.51,<5", "polars>=1.44,<2", "scikit-learn>=1.6,<2",
    ])

if REPO_PATH is not None:
    root = Path(REPO_PATH).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working")]
    candidates += [p.parents[2] for p in Path("/kaggle/working").glob("*/src/retrievers/dense.py")]
    root = next((p for p in candidates if (p / "src/retrievers/dense.py").exists()), None)
    if root is None and REPO_URL:
        root = Path("/kaggle/working") / Path(REPO_URL.removesuffix(".git")).name
        if not root.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(root)])
    if root is None:
        raise FileNotFoundError("Не найден код проекта. Укажите REPO_PATH или REPO_URL.")
if not (root / "src/retrievers/dense.py").exists():
    raise FileNotFoundError(f"Нет src/retrievers/dense.py в {root}. Обновите репозиторий или REPO_PATH.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Код:", root)

## Данные и сплит

In [ ]:
import gc
import numpy as np
import pandas as pd
import polars as pl
import torch
from IPython.display import display
from src.split import split_queries
from src.retrievers.dense import DenseRetriever

if not torch.cuda.is_available():
    raise RuntimeError("GPU не обнаружена. В настройках Kaggle включите Accelerator: GPU.")
print("GPU:", torch.cuda.get_device_name(0))

if DATA_PATH is not None:
    data_path = Path(DATA_PATH)
else:
    local_path = root / "data/train.parquet"
    if local_path.is_file():
        data_path = local_path
    else:
        matches = sorted(Path("/kaggle/input").rglob("train.parquet"))
        if len(matches) != 1:
            raise FileNotFoundError(f"Найдено {len(matches)} файлов train.parquet в /kaggle/input: {matches}. Укажите DATA_PATH.")
        data_path = matches[0]
print("Данные:", data_path)

QUERY_COLUMNS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]
interactions = pl.read_parquet(data_path, columns=QUERY_COLUMNS + ["item_id"])
interactions = interactions.with_columns(
    pl.Series("split", split_queries(interactions["search_query"], seed=SEED))
)
items = (
    pl.scan_parquet(data_path).select("item_id", "item_title_raw")
    .unique(subset="item_id", keep="last", maintain_order=True)
    .sort("item_id").collect()
)
corpus = pd.DataFrame(items.to_dict(as_series=False))
selected = (
    interactions.filter(pl.col("split") == "test")
    .group_by(QUERY_COLUMNS).agg(pl.col("item_id").unique().alias("targets"))
    .sort(QUERY_COLUMNS)
)
if N_TEST_QUERIES is not None:
    selected = selected.sample(n=min(N_TEST_QUERIES, len(selected)), seed=SEED, shuffle=True)
queries = pd.DataFrame(selected.select(QUERY_COLUMNS).to_dict(as_series=False))
targets = [set(ids) for ids in selected["targets"].to_list()]
assert all(targets)
assert set().union(*targets).issubset(set(corpus["item_id"]))
print(f"Объявлений: {len(corpus):,}; тестовых контекстов: {len(queries):,}")
display(interactions.group_by("split").len().sort("split"))
del interactions, items, selected
gc.collect()

## Recall@K

Для каждого тестового контекста считаем долю его уникальных наблюдаемых `item_id` в top-K. Затем усредняем по всем контекстам; отсутствие выдачи даёт 0. Это та же самописная метрика, что в BM25/TF-IDF ноутбуках.

In [ ]:
def recall_at_k(candidates, targets, k):
    retrieved = (
        candidates.loc[candidates["rank"] <= k]
        .groupby("query_index")["item_id"].agg(set).to_dict()
    )
    return float(np.mean([
        len(relevant & retrieved.get(i, set())) / len(relevant)
        for i, relevant in enumerate(targets)
    ]))

example = pd.DataFrame({"query_index": [0, 0, 0], "item_id": ["a", "a", "b"], "rank": [1, 2, 3]})
assert recall_at_k(example, [{"a", "b"}, {"c"}], k=2) == 0.25

## Общий запуск модели

Индекс хранит нормированные векторы заголовков. При повторном запуске он загружается с диска, если совпадает список `item_id`. Если тексты изменились при прежних ID, задайте `FORCE_REBUILD=True`. После каждого эксперимента модель освобождается из GPU.

In [ ]:
from time import perf_counter

index_root = Path(INDEX_DIR)
index_root.mkdir(parents=True, exist_ok=True)
summary = []

def run_model(name):
    path = index_root / name
    if not FORCE_REBUILD and (path / "index.faiss").is_file() and (path / "metadata.json").is_file():
        model = DenseRetriever.load(path, device="cuda")
        if model.model != name or model.max_length != MAX_LENGTH:
            raise ValueError(f"Индекс {path} создан с другими настройками. Задайте FORCE_REBUILD=True.")
        if model.item_ids_.tolist() != corpus["item_id"].loc[corpus["item_title_raw"].fillna("").str.strip().ne("")].tolist():
            raise ValueError(f"Корпус для {path} изменился. Задайте FORCE_REBUILD=True.")
        model.batch_size = BATCH_SIZE
        print(f"{name}: загружен индекс {path}")
    else:
        model = DenseRetriever(model=name, device="cuda", batch_size=BATCH_SIZE,
                               max_length=MAX_LENGTH, show_progress=True)
        started = perf_counter()
        model.fit(corpus)
        print(f"{name}: индекс построен за {perf_counter() - started:.1f} с")
        model.save(path)
        print(f"{name}: индекс сохранён в {path}")
    started = perf_counter()
    candidates = model.retrieve(queries, k=max(KS))
    print(f"{name}: поиск выполнен за {perf_counter() - started:.1f} с")
    assert candidates["query_index"].between(0, len(queries) - 1).all()
    assert not candidates.duplicated(["query_index", "item_id"]).any()
    assert np.isfinite(candidates["score"]).all()
    row = {"Модель": name, "Кандидатов": len(candidates)}
    row.update({f"Recall@{k}": recall_at_k(candidates, targets, k) for k in KS})
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return row, candidates

## 1. BGE-M3

In [ ]:
bge_result, bge_candidates = run_model("bge")
display(pd.DataFrame([bge_result]))
display(bge_candidates.head(10))
summary.append(bge_result)
del bge_candidates
gc.collect()

## 2. Giga-Embeddings 480M

In [ ]:
giga_result, giga_candidates = run_model("giga")
display(pd.DataFrame([giga_result]))
display(giga_candidates.head(10))
summary.append(giga_result)
del giga_candidates
gc.collect()

## Сравнение

In [ ]:
comparison = pd.DataFrame(summary).set_index("Модель")
display(comparison)
print("Индексы:", index_root)